<a href="https://colab.research.google.com/github/Hyeryeong16/Python_CLI_board/blob/main/%ED%8C%8C%EC%9D%B4%EC%8D%AC_CLI_%EA%B2%8C%EC%8B%9C%ED%8C%90_%EB%A7%8C%EB%93%A4%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from datetime import datetime
from dataclasses import dataclass # 더 간결한 표현이 가능하게 해줌 @dataclass
# ===== 공통 함수 (중복 제거) =====

@dataclass
class Article:
  id : int
  regDate : str
  updateDate : str
  memberId : int
  title : str
  body : str

@dataclass
class Member:
  id : int
  regDate : str
  updateDate : str
  loginId : str
  loginPw : str
  name : str

def now():
    # 현재 시각을 "YYYY-MM-DD HH:MM:SS" 문자열로
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def parse_id(cmd):
    # "article delete 3" -> 3 을 뽑아준다. 잘못된 명령이면 None
    cmdBits = cmd.split(" ")
    if len(cmdBits) < 3:
        print("명령어를 확인해주세요(길이 모자람)")
        return None
    if not cmdBits[2].isdigit():
        print("명령어를 확인해주세요(번호대신 str 입력)")
        return None
    return int(cmdBits[2])

def find_article(articles, article_id):
    # id가 맞는 글을 돌려준다. 없으면 None (found 플래그가 필요 없어짐)
    for article in articles:
        if article.id == article_id:
            return article
    return None

def format_date(regDate):
    # 오늘 쓴 글이면 시:분:초, 이전 글이면 연-월-일 로 보여준다
    today = now().split(" ")[0]        # "2026-06-30"
    datePart = regDate.split(" ")[0]   # 글의 날짜
    timePart = regDate.split(" ")[1]   # 글의 시각
    if datePart == today:
        return timePart
    else:
        return datePart

def make_articleTestData():
  # 시작하자마자 넣어둘 테스트용 글 3개를 만들어서 리스트로 리턴한다
  # 1번은 이전 날짜 -> 목록 날짜 표기 테스트용
  return [
      Article(1, "2025-12-12 12:12:12","2025-12-12 12:12:12", 1, "제목1", "내용1"),
      Article(2, now(), now(), 1, "제목2", "내용2"),
      Article(3, now(), now(), 2, "제목3", "내용3"),
          ]

def make_memberTestData():
  return [
      Member(1, now(), now(), "id1", "pw1", "name1"),
      Member(2, now(), now(), "id2", "pw2", "name2"),
      Member(3, now(), now(), "id3", "pw3","name3")
  ]

def find_loginid(members, login_id):
    for member in members:
        if member.loginId == login_id:
            return None # 같은 아이디가 있음
    return login_id # 아이디 없음

def find_loginPw(members,login_id, login_pw):
    for member in members:
      if member.loginId == login_id and member.loginPw == login_pw:
        return member # 비번 맞음
    return None # 비번 틀림

# ===== 메인 =====

print("== CLI 게시판 실행 ==")

last_article_id = 3
articles = make_articleTestData()
last_member_id = 3
members = make_memberTestData()

loginedMember = None

while True:
    cmd = input("명령어 ) ").strip()

    if cmd == "exit":
        break

    elif cmd == "member join":
      if loginedMember is not None:
        print("이미 로그인 했습니다")
        continue

      last_member_id += 1
      while True:
        loginId = input("loginId : ")
        check = find_loginid(members,loginId)
        if check is None: # 같은 아이디 있어
          print("이미 사용 중인 아이디입니다. 다시 입력해주세요.")
          continue
        print("사용 가능한 아이디입니다.")
        break

      while True:
        loginPw = input("loginPw : ")
        loginPwConfirm = input("loginPwConfirm : ")
        if loginPwConfirm != loginPw:
          print("비밀번호가 맞지 않습니다.")
          continue
        break

      name = input("name : ")
      member = Member(last_member_id, now(), now(), loginId, loginPw, name)
      members.append(member)
      print("{}번 회원이 가입되었습니다".format(last_member_id))
      print(members)

    elif cmd == "member login":
      if loginedMember is not None:
        print("이미 로그인 했습니다")
        continue

      while True:
        loginId = input("loginId : ")
        check_id = find_loginid(members,loginId)

        while True:
          if check_id is not None: # id 없을 때
            print("없는 ID입니다.")
            break
          else:
            loginPw = input("loginPw : ") # id 있을 때
            check_pw = find_loginPw(members, loginId, loginPw)
            if check_pw is None: # pw 틀렸을 때
              print("틀린 비밀번호입니다. 다시 입력해주세요.")
              continue
            else: # pw 맞을 때
              print("로그인 했습니다")
              loginedMember = check_pw # member를 return해서 회원의 정보를 가지고 있음
              break
        break

    elif cmd == "member logout":
      if loginedMember is None:
        print("로그인 후 이용가능")
        continue

      print("로그아웃 되었습니다")
      loginedMember = None

    elif cmd == "article write":
      if loginedMember is None:
        print("로그인 후 이용가능")
        continue

      last_article_id += 1
      title = input("제목 : ")
      body = input("내용 : ")
      article = Article(last_article_id, now(), now(), loginedMember.id, title, body)
      articles.append(article)
      print("{}번 글이 생성되었습니다".format(last_article_id))

    elif cmd == "article list":
      if not articles:
        print("게시글이 없습니다")
      else:
        print("=========================================")
        print("   번호      /     제목       /     내용   /     작성일   ")
        for a in reversed(articles):
          print("   {}      /     {}       /     {}          /     {}   ".format(
              a.id, a.title, a.body, format_date(a.regDate)))
        print("=========================================")

    elif cmd.startswith("article delete"):
      deletedId = parse_id(cmd)
      if deletedId is None:
        continue

      if loginedMember is None:
        print("로그인 후 이용가능")
        continue

      article = find_article(articles, deletedId)
      if article is None:
        print("{}번 글은 없습니다".format(deletedId))
        continue

      if article.memberId != loginedMember.id: # 글의 작성자가 로그인한 회원이 아니면 삭제할 수 없게
          print("권한이 없습니다")
          continue

      articles.remove(article)
      print("{}번 글이 삭제되었습니다".format(deletedId))

    elif cmd.startswith("article edit"):
      editedId = parse_id(cmd)
      if editedId is None:
        continue

      article = find_article(articles, editedId)
      if article is None:
        print("{}번 글은 없습니다".format(editedId))
        continue

      if article.memberId != loginedMember.id:
          print("권한이 없습니다")
          continue

      print("기존 제목 : {}".format(article.title))
      print("기존 내용 : {}".format(article.body))
      article.title = input("새 제목 : ")
      article.body = input("새 내용 : ")
      article.updateDate = now()
      print("{}번 글이 수정되었습니다".format(editedId))

    elif cmd.startswith("article detail"):
      detailId = parse_id(cmd)
      if detailId is None:
        continue

      if loginedMember is None:
        print("로그인 후 이용가능")
        continue

      article = find_article(articles, detailId)
      if article is None:
        print("{}번 글은 없습니다".format(detailId))
        continue

      print("번호 : {}".format(article.id))
      print("작성 날짜 : {}".format(article.regDate))
      print("수정 날짜 : {}".format(article.updateDate))
      print("제목 : {}".format(article.title))
      print("내용 : {}".format(article.body))

    else:
        print("지원하지 않는 기능입니다")

print("== CLI 게시판 종료 ==")



== CLI 게시판 실행 ==
명령어 ) member login
loginId : id1
loginPw : pw1
로그인 했습니다
명령어 ) article write
제목 : q
내용 : a
4번 글이 생성되었습니다
명령어 ) article edit 3
권한이 없습니다
명령어 ) article edit 1
기존 제목 : 제목1
기존 내용 : 내용1
새 제목 : as
새 내용 : zx
1번 글이 수정되었습니다
명령어 ) article delete 4
4번 글이 삭제되었습니다
명령어 ) article delete 3
권한이 없습니다
명령어 ) member logout
로그아웃 되었습니다
명령어 ) article write
로그인 후 이용가능
명령어 ) exit
== CLI 게시판 종료 ==
